# 03b · Preprocesamiento — de las decisiones del EDA a código con pruebas

> Continúa `01b_caso_negocio_aprueba_o_no.ipynb` (**aprueba con 70 o más; con menos de 70 pierde el curso**; clase de interés `Fail`, 69,2 %) y `02b_eda_aprueba_o_no.ipynb` (tabla de decisiones).
> Especificación: `especificaciones/03-preprocesamiento.md`. Dataset: `data/student_exam_performance.csv`. Plantilla: `notebooks/03_preprocesamiento.ipynb`.

Cada decisión del EDA es una **función pequeña** en `src/proyecto/data/` con **pruebas** en `tests/`. Este notebook **solo orquesta**: llama a las funciones y muestra resultados; no contiene lógica reutilizable.

> **El objetivo es `pass_70`**, creado desde `exam_score` con el corte en 70 (`add_target`): `Pass` si la nota es ≥ 70 y `Fail` si es < 70 (69 y 69,99 pierden el curso).
> La columna `pass_status` del dataset usa el corte en **50**: **no es el objetivo** y se excluye como fuga.

| Decisión del EDA (02b) | Qué se hace en esta etapa | Función |
|---|---|---|
| 1. 6 columnas con la respuesta o del examen | Se **crea `pass_70`** y después se quitan: `exam_score`, `pass_status`, `performance_grade`, `performance_level`, `questions_correct`, `questions_attempted` | `add_target`, `drop_leakage_columns` |
| 2. Vacíos en 7 columnas (41,9 % de las filas) | **No se borra nada**; solo se informa. Se imputan **dentro del `Pipeline`** (etapa 4) | `check_missing_report` |
| 3. `student_id` único → partir por fila | `student_id` pasa al índice; partición 80/20 estratificada por `pass_70`, semilla 42 | `drop_identifier`, `split_train_test` |
| 4. Métricas: costo, *recall* de `Fail` y NPV | No aplica aquí (etapas 5 a 7); la estratificación mantiene el 69,2 % de `Fail` en ambos lados | `split_train_test` |
| 5 a 7. Topes, horas y señal | Sin cambios: se conservan las variables y se prueban en la etapa 4 | — |
| 8. `family_income` y `gender` sensibles | Salen de las variables de entrada y se guardan **aparte** para auditar la equidad | `drop_sensitive_columns`, `extract_audit_columns` |
| 9. Variables «a medir con y sin» | `time_management_score`, `exam_anxiety_level` y `exam_difficulty` **se conservan** hasta la etapa 4 | — |
| Rangos lógicos (defensivo) | El EDA no encontró valores imposibles; la regla marca como vacío lo que lo fuera en datos nuevos | `mark_out_of_range` |

**Regla de oro:** `data/` (el CSV original) **no se modifica**; el notebook lo comprueba con una huella digital antes y después.

In [ ]:
import sys
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore")
# Permite importar tu paquete de src/ sin instalarlo
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

from proyecto.data.cleaning import (ID_COL, LEAK_COLUMNS, PASS_THRESHOLD, TARGET, check_missing_report, clean,
                                    drop_sensitive_columns, extract_audit_columns, load_raw)
from proyecto.data.split import split_train_test

TEST_SIZE = 0.2    # 20 % para la prueba (cerrada hasta la etapa 7)
SEED = 42

## 0. Cargar los datos crudos

In [ ]:
import hashlib

CANDIDATES = [Path.cwd().resolve().parent / "data" / "raw" / "student_exam_performance.csv",
              Path.cwd().resolve().parent / "data" / "student_exam_performance.csv"]
RAW_PATH = next((p for p in CANDIDATES if p.exists()), None)
assert RAW_PATH is not None, "No encontré student_exam_performance.csv en data/raw/ ni en data/"

def fingerprint(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

raw_hash_before = fingerprint(RAW_PATH)          # para demostrar al final que el original no cambió
df = load_raw(RAW_PATH)                          # student_id queda como índice
print(f"{RAW_PATH.name}: {df.shape[0]:,} filas x {df.shape[1]} columnas (sin contar el índice student_id)")
print(f"con el corte en {PASS_THRESHOLD}: {(df['exam_score'] < PASS_THRESHOLD).mean():.1%} pierde el curso (Fail); "
      f"con el corte de 50 del dataset (pass_status) sería {(df['pass_status'] == 'Fail').mean():.1%}")
# Datos personales: no hay nombres ni documentos. No se imprime ninguna fila de estudiantes.

## 1. Limpieza

`clean(df)` aplica, en orden: `student_id` al índice → **crea el objetivo `pass_70`** desde `exam_score` → quita las 6 columnas de fuga → marca como vacío lo imposible.
**El orden importa:** el objetivo se crea *antes* de quitar `exam_score`. **No borra filas** y **no modifica `data/raw/`**.

In [ ]:
clean_df = clean(df)
quitadas = [c for c in df.columns if c not in clean_df.columns]
nuevas = [c for c in clean_df.columns if c not in df.columns]
print(f"antes: {df.shape}  ->  después: {clean_df.shape}")
print("columnas quitadas:", quitadas)
print("columnas nuevas:", nuevas)
print("\nVacíos que quedan (se resuelven dentro del Pipeline, etapa 4):")
display(check_missing_report(clean_df))

## 2. Comprobaciones de la limpieza

Cada decisión del EDA se convierte en una **afirmación que falla si algo no se cumple** (no es un gráfico).

In [ ]:
vacios_antes = df.drop(columns=list(LEAK_COLUMNS)).isna().sum().sum()
perdedores = df["exam_score"] < PASS_THRESHOLD
checks = {
    "no se perdieron filas": len(clean_df) == len(df) == 100_000,
    "las 6 columnas de fuga ya no están (incluida pass_status, corte en 50)": not set(LEAK_COLUMNS) & set(clean_df.columns),
    "el objetivo pass_70 existe, sin vacíos y solo con Pass / Fail": TARGET in clean_df.columns and set(clean_df[TARGET].unique()) == {"Pass", "Fail"},
    f"pass_70 cumple la regla: Fail ⇔ nota < {PASS_THRESHOLD} (69 pierde, 70 aprueba)": bool(((clean_df[TARGET] == "Fail") == perdedores).all()),
    "hay 69.210 Fail y 30.790 Pass, como midió el EDA": (clean_df[TARGET] == "Fail").sum() == 69_210 and (clean_df[TARGET] == "Pass").sum() == 30_790,
    "student_id no es una variable de entrada": ID_COL not in clean_df.columns and clean_df.index.is_unique,
    "la limpieza no creó vacíos nuevos (el EDA no halló valores imposibles)": clean_df.isna().sum().sum() == vacios_antes,
    "los vacíos siguen en las 7 columnas del EDA": set(check_missing_report(clean_df).index) == {
        "attendance_percentage", "time_management_score", "notes_quality", "previous_gpa",
        "sleep_quality", "parent_education", "device_availability"},
    "las variables 'a medir con y sin' se conservan": {"time_management_score", "exam_anxiety_level", "exam_difficulty"} <= set(clean_df.columns),
}
result = pd.Series(checks, name="se cumple")
assert result.all(), result[~result]
display(result)

## 2b. Atributos sensibles: fuera del modelo, guardados aparte

Decisión 8 del EDA: `family_income` (brecha de **15 puntos** en `Fail`: 75,5 % con ingreso bajo y 60,5 % con alto) y `gender` **no entran como predictores**. Se copian antes a una tabla de **auditoría** con el mismo índice, para comprobar después si el modelo falla más con algún grupo.

In [ ]:
audit = extract_audit_columns(clean_df)          # gender, family_income, urban_rural, school_type, education_level, age
features = drop_sensitive_columns(clean_df)      # lo que verá el modelo

checks = {
    "gender y family_income no están en las variables de entrada": not {"gender", "family_income"} & set(features.columns),
    "la tabla de auditoría tiene los mismos estudiantes": audit.index.equals(features.index),
    "la auditoría conserva los grupos a revisar": {"gender", "family_income", "urban_rural", "school_type"} <= set(audit.columns),
    "student_id no está entre las columnas": ID_COL not in features.columns and ID_COL not in audit.columns,
}
result = pd.Series(checks, name="se cumple")
assert result.all(), result[~result]
print(f"variables de entrada: {features.shape[1] - 1} (más el objetivo) | columnas de auditoría: {list(audit.columns)}")

## 3. Partición por la unidad de análisis

`student_id` es único: **cada fila es un estudiante distinto**, así que partir **por fila** es válido. La partición se **estratifica por `pass_70`** para que entrenamiento y prueba tengan la misma proporción de `Fail` (69,2 %), y es **reproducible** (semilla 42).

In [ ]:
train, test = split_train_test(features, target=TARGET, test_size=TEST_SIZE, seed=SEED)

assert set(train.index).isdisjoint(set(test.index)), "hay estudiantes en los dos lados"
assert len(train) + len(test) == len(features), "se perdieron filas"
fail_train, fail_test = (train[TARGET] == "Fail").mean(), (test[TARGET] == "Fail").mean()
assert abs(fail_train - fail_test) < 0.005, "la proporción de Fail difiere entre los dos lados"

# La auditoría se parte con los MISMOS estudiantes (misma partición, sin volver a sortear)
audit_train, audit_test = audit.loc[train.index], audit.loc[test.index]

print(f"entrenamiento: {len(train):,} filas · prueba: {len(test):,} filas (cerrada hasta la etapa 7)")
print(f"% de Fail  ->  entrenamiento {fail_train:.1%} | prueba {fail_test:.1%} | total {(features[TARGET] == 'Fail').mean():.1%}")

**Mi unidad de análisis y por qué parto así:** la unidad es **un estudiante por fila** y ninguno se repite (`student_id`: 100.000 ids para 100.000 filas), por lo que **partir por fila es válido**: un mismo estudiante nunca queda en entrenamiento y prueba a la vez.
La estratificación mantiene el 69,2 % de `Fail` en ambos lados, para que las métricas (costo, *recall* de `Fail`, NPV) sean comparables.

> Si algún dato futuro trajera a un mismo estudiante en varias filas, **esta partición dejaría de ser válida** y habría que partir por grupo. Por eso `split_train_test` falla si el índice tiene valores repetidos.

## 3b. Lo que hereda la etapa 4 (variables y vacíos)

Imputar con medianas o categorías **depende del entrenamiento**, así que **no se hace aquí**: va dentro del `Pipeline`, ajustado solo con `train`. Este cuadro deja escrito el plan (de `especificaciones/03-preprocesamiento.md`) y comprueba que cubre exactamente las columnas con vacíos.

In [ ]:
X_cols = [c for c in train.columns if c != TARGET]
numeric = [c for c in X_cols if pd.api.types.is_numeric_dtype(train[c])]
categorical = [c for c in X_cols if c not in numeric]
print(f"{len(X_cols)} variables de entrada: {len(numeric)} numéricas y {len(categorical)} categóricas")

plan_vacios = {
    "parent_education": "categoría explícita 'Desconocido'",
    "notes_quality": "categoría explícita 'Desconocido'",
    "sleep_quality": "categoría explícita 'Desconocido'",
    "device_availability": "categoría explícita 'Desconocido'",
    "previous_gpa": "mediana del entrenamiento + indicador de vacío",
    "attendance_percentage": "mediana del entrenamiento + indicador de vacío",
    "time_management_score": "mediana del entrenamiento + indicador de vacío",
}
assert set(plan_vacios) == set(check_missing_report(train).index), "el plan de vacíos no coincide con las columnas que tienen vacíos"
display(check_missing_report(train).assign(tratamiento_en_la_etapa_4=pd.Series(plan_vacios)))

## 4. Guardar los datos de la etapa

Se guardan en `data/processed/` (**no se versionan**: se regeneran con este notebook; la carpeta está en `.gitignore`). La prueba **no se vuelve a mirar** hasta la validación final (etapa 7).
Se guardan también las tablas de **auditoría**, alineadas con los mismos estudiantes.

In [ ]:
import joblib

OUT = Path.cwd().resolve().parent / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)
saved = {"train": train, "test": test, "audit_train": audit_train, "audit_test": audit_test}
for name, frame in saved.items():
    joblib.dump(frame, OUT / f"{name}.joblib")

# Comprobación de ida y vuelta (solo forma e índice: la prueba no se mira)
for name, frame in saved.items():
    back = joblib.load(OUT / f"{name}.joblib")
    assert back.shape == frame.shape and back.index.equals(frame.index), name
print("guardado en data/processed/:", ", ".join(f"{n}.joblib {f.shape}" for n, f in saved.items()))

# El CSV original no cambió
assert fingerprint(RAW_PATH) == raw_hash_before, "¡el CSV original cambió!"
print("el CSV original NO cambió (misma huella digital antes y después)")

## 5. Las pruebas automáticas

Desde la raíz del repo: `pytest -q`. Hay pruebas para cada función de limpieza (`tests/test_cleaning.py`, incluida la regla del 70: **69 pierde y 70 aprueba**) y para la partición (`tests/test_split.py`).
La celda siguiente las ejecuta desde aquí (si `pytest` está instalado).

In [ ]:
import subprocess

root = Path.cwd().resolve().parent
run = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_cleaning.py", "tests/test_split.py", "-p", "no:cacheprovider"],
                     cwd=root, capture_output=True, text=True)
print((run.stdout or run.stderr).strip()[-600:])

## Entregable de esta etapa (hito H1)

- [ ] La limpieza corre de principio a fin desde el repo.
- [ ] El objetivo `pass_70` cumple la regla (nota ≥ 70 aprueba; 69 pierde) y `pass_status` (corte en 50) queda fuera.
- [ ] La partición respeta mi unidad de análisis (por fila, porque `student_id` es único) y está **verificada con una prueba**.
- [ ] `pytest` en verde.
- [ ] `data/raw/` (el CSV original) no cambió.
- [ ] Declaré mi unidad de análisis por escrito.
- [ ] Probé que las pruebas **pueden fallar** (por ejemplo, usar `> 70` en vez de `>= 70`, partir sin estratificar o dejar un `student_id` repetido) y volví a dejar el código bien.
- [ ] El grupo confirmó que `family_income` y `gender` quedan fuera de las variables de entrada.

**Siguiente:** `04_features.ipynb` (variables y pliegues), usando `data/processed/train.joblib`. **No abrir `test.joblib` hasta la etapa 7.**